# CQAS Detection Accuracy Experiment

Experiments to measure detection accuracy (DAR) and false positive rate (FPR) under varying conditions.

In [ ]:
import sys
sys.path.insert(0, '..')
import matplotlib.pyplot as plt
import numpy as np
from src.detection.rule_engine import RuleEngine
from src.validation.iqc_engine import IQCEngine, SYNTHETIC_IOCS
from src.metrics.performance_metrics import PerformanceMetrics
from src.metrics.scoring import ScoringEngine

## 1. Run IQC Validation Cycle

In [ ]:
rule_engine = RuleEngine('../configs/siem_rules.yaml')
iqc = IQCEngine(rule_engine)
results = iqc.run_all()
summary = iqc.summary()
print(f'IQC Results: {summary["passed"]}/{summary["total"]} passed')
print(f'DAR: {summary["dar_percent"]:.1f}%')

## 2. DAR by IOC Type

In [ ]:
ioc_names = [r['ioc_name'] for r in summary['results']]
pass_fail = [1 if r['passed'] else 0 for r in summary['results']]
colors = ['green' if p else 'red' for p in pass_fail]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(ioc_names, pass_fail, color=colors, alpha=0.8)
ax.set_xlabel('Detected (1=Pass, 0=Fail)')
ax.set_title(f'IQC Detection Results — DAR: {summary["dar_percent"]:.1f}%')
ax.set_xlim(0, 1.2)
for bar, val in zip(bars, pass_fail):
    label = 'PASS' if val else 'FAIL'
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, label,
            va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Threshold Sensitivity Analysis

In [ ]:
# Simulate how FPR changes with different thresholds
thresholds = list(range(3, 25))
simulated_fprs = [max(0, 30 - t*1.5 + np.random.normal(0, 1)) for t in thresholds]
simulated_dars = [min(100, 70 + t*1.2 - np.random.normal(0, 2)) for t in thresholds]

fig, ax1 = plt.subplots(figsize=(10, 6))
color1, color2 = 'steelblue', 'tomato'
ax1.plot(thresholds, simulated_dars, color=color1, marker='o', label='DAR %', linewidth=2)
ax1.set_xlabel('Detection Threshold (count)')
ax1.set_ylabel('DAR (%)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(95, color=color1, linestyle='--', alpha=0.5, label='DAR target (95%)')

ax2 = ax1.twinx()
ax2.plot(thresholds, simulated_fprs, color=color2, marker='s', label='FPR %', linewidth=2)
ax2.set_ylabel('FPR (%)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)
ax2.axhline(5, color=color2, linestyle='--', alpha=0.5, label='FPR target (5%)')

plt.title('Detection Threshold Sensitivity: DAR vs. FPR Trade-off')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.show()

## 4. Quality Score Computation

In [ ]:
perf = PerformanceMetrics()
scorer = ScoringEngine()
metrics = perf.full_report(iqc_results=results)
quality = scorer.score_from_metrics_report(metrics)
print(f'Quality Score: {quality.composite_score:.1f}/100 (Grade: {quality.grade})')
for component, score in quality.component_scores.items():
    print(f'  {component}: {score:.1f}')